# 01 - Organize and extract lunar raw data

Run all cells in order with Python 3.10+ (standard library only), from the project root or its notebooks folder. Set PROJECT_ROOT explicitly if running elsewhere. Stop download processes writing to these folders before running.

**Raw data: source first, then source-provided location folders.** Shared regional maps remain single copies. Future processed outputs should use `data/processed/moon/<region_id>/<grid_id>/`, with aligned feature layers and provenance together. No processing or resampling occurs here.

This notebook preserves all ZIPs and scientific filenames. It moves the seven regional archives into `data/raw/moon/sdem/archives`, extracts their original region folders under `sdem/regions`, and groups LOLA Product 90 files and manifests under `lola_product90`. Diviner and illumination stay in their existing folders. ZIP entries under `__MACOSX`, `.DS_Store`, or AppleDouble `._*` are skipped during extraction, but remain in the preserved ZIPs.

Preflight checks reject path traversal, symlinks, conflicting destinations, and insufficient space. Extraction verifies ZIP CRCs; reruns validate existing extracted files against CRC and size. An interrupted extraction leaves `.extracting` files that the next run replaces. No original data is deleted. Each completed run writes a timestamped audit JSON under `data/manifests/organization/`.
The initial move plan is saved before moves begin; the audit is updated after each move and archive and marked complete only after final verification. Earlier run audits are retained.


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import json
import shutil
import stat
import zipfile
import zlib

discovered_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "outline.md").is_file()),
    None,
)
if discovered_root is None:
    raise RuntimeError("Set PROJECT_ROOT to the folder containing outline.md.")
PROJECT_ROOT: Path = discovered_root.resolve()
MOON = PROJECT_ROOT / "data/raw/moon"
SDEM = MOON / "sdem"
LOLA = MOON / "lola_product90"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
audit = {"run_id": RUN_ID, "moves": [], "archives": [], "excluded_entries": []}

def checked_path(path: Path) -> Path:
    resolved = path.resolve()
    if not resolved.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"Path outside workspace: {path}")
    return resolved


def crc32_file(path: Path) -> int:
    crc = 0
    with path.open("rb") as handle:
        while chunk := handle.read(8 * 1024 * 1024):
            crc = zlib.crc32(chunk, crc)
    return crc & 0xFFFFFFFF


def members(archive: zipfile.ZipFile, region: str):
    for entry in archive.infolist():
        relative = PurePosixPath(entry.filename)
        if relative.is_absolute() or ".." in relative.parts or "\\" in entry.filename or ":" in entry.filename:
            raise ValueError(f"Unsafe ZIP member: {entry.filename}")
        if stat.S_ISLNK(entry.external_attr >> 16):
            raise ValueError(f"ZIP symlink is unsupported: {entry.filename}")
        if any(p == "__MACOSX" or p == ".DS_Store" or p.startswith("._") for p in relative.parts):
            continue
        if not relative.parts or relative.parts[0] != region:
            raise ValueError(f"Unexpected archive root: {entry.filename}")
        if not entry.is_dir():
            yield entry, checked_path(SDEM / "regions" / Path(*relative.parts))

archives = sorted([*MOON.glob("A3CLR22_*.zip"), *(SDEM / "archives").glob("A3CLR22_*.zip")])
if not archives:
    raise RuntimeError("No SDEM archives found in the original or organized location.")
if len({p.name for p in archives}) != len(archives):
    raise RuntimeError("Duplicate archive names across old and new folders; resolve before running.")
required_bytes = 0
for source in archives:
    with zipfile.ZipFile(source) as archive:
        entries = list(members(archive, source.stem))
        destinations = [str(destination).casefold() for _, destination in entries]
        if len(destinations) != len(set(destinations)):
            raise ValueError(f"Duplicate extraction paths in {source.name}")
        required_bytes += sum(entry.file_size for entry, destination in entries if not destination.exists())
        print(f"{source.name}: {len(entries)} data files")
free_bytes = shutil.disk_usage(PROJECT_ROOT).free
if free_bytes < required_bytes + 1024**3:
    raise RuntimeError("Insufficient disk space including a 1 GiB reserve.")
print(f"New extraction space: {required_bytes / 1e9:.2f} GB; free: {free_bytes / 1e9:.2f} GB")

move_pairs = [(p, SDEM / "archives" / p.name) for p in archives if p.parent == MOON]
for name in ("large_area_dem", "roughness", "permanent_shadow", "nac_images", "point_clouds", "other",
             "download_manifest.csv", "source_page_snapshot.html", "failed_downloads.csv"):
    source = MOON / name
    if source.exists():
        move_pairs.append((source, LOLA / name))
for source, destination in move_pairs:
    checked_path(source)
    checked_path(destination)
    if destination.exists():
        raise FileExistsError(f"Refusing to merge or overwrite: {destination}")
    print(f"MOVE {source.relative_to(PROJECT_ROOT)} -> {destination.relative_to(PROJECT_ROOT)}")

In [ ]:
# Persist the plan before moving data, then checkpoint each completed operation.
report_dir = checked_path(PROJECT_ROOT / "data/manifests/organization")
report_dir.mkdir(parents=True, exist_ok=True)
report = report_dir / f"{RUN_ID}.json"
audit["status"] = "in_progress"
audit["planned_moves"] = [{"from": str(s.relative_to(PROJECT_ROOT)), "to": str(d.relative_to(PROJECT_ROOT))} for s, d in move_pairs]

def save_audit() -> None:
    temporary_report = report.with_suffix(".json.tmp")
    temporary_report.write_text(json.dumps(audit, indent=2), encoding="utf-8")
    temporary_report.replace(report)

save_audit()

# Same-volume renames preserve original file contents, including retained .part files.
for source, destination in move_pairs:
    before = {str(p.relative_to(source)): p.stat().st_size for p in source.rglob("*") if p.is_file()} if source.is_dir() else {"file": source.stat().st_size}
    destination.parent.mkdir(parents=True, exist_ok=True)
    source.rename(destination)
    after = {str(p.relative_to(destination)): p.stat().st_size for p in destination.rglob("*") if p.is_file()} if destination.is_dir() else {"file": destination.stat().st_size}
    if before != after:
        raise RuntimeError(f"Move verification failed: {destination}")
    audit["moves"].append({"from": str(source.relative_to(PROJECT_ROOT)), "to": str(destination.relative_to(PROJECT_ROOT)), "files": before})
    save_audit()
    print(f"Moved and checked: {destination.relative_to(PROJECT_ROOT)}")

for source in sorted((SDEM / "archives").glob("A3CLR22_*.zip")):
    records = []
    with zipfile.ZipFile(source) as archive:
        entries = list(members(archive, source.stem))
        included = {entry.filename for entry, _ in entries}
        audit["excluded_entries"].extend({"archive": source.name, "entry": entry.filename} for entry in archive.infolist() if not entry.is_dir() and entry.filename not in included)
        for entry, destination in entries:
            if destination.exists():
                if destination.stat().st_size != entry.file_size or crc32_file(destination) != entry.CRC:
                    raise RuntimeError(f"Existing extracted file differs from archive: {destination}")
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                temporary = checked_path(destination.with_name(destination.name + ".extracting"))
                with archive.open(entry) as incoming, temporary.open("wb") as outgoing:
                    shutil.copyfileobj(incoming, outgoing, length=8 * 1024 * 1024)
                if temporary.stat().st_size != entry.file_size or crc32_file(temporary) != entry.CRC:
                    raise RuntimeError(f"Extraction verification failed: {temporary}")
                temporary.replace(destination)
            records.append({"path": str(destination.relative_to(PROJECT_ROOT)), "bytes": entry.file_size, "crc32": f"{entry.CRC:08x}"})
    audit["archives"].append({"archive": str(source.relative_to(PROJECT_ROOT)), "bytes": source.stat().st_size, "files": records})
    save_audit()
    print(f"Extracted/verified {source.stem}: {len(records)} files", flush=True)

In [ ]:
# Verify all recorded outputs again by presence and size; CRC was checked above.
for archive_record in audit["archives"]:
    for record in archive_record["files"]:
        path = checked_path(PROJECT_ROOT / record["path"])
        if not path.is_file() or path.stat().st_size != record["bytes"]:
            raise RuntimeError(f"Missing or changed output: {path}")
unfinished = list((SDEM / "regions").rglob("*.extracting"))
if unfinished:
    raise RuntimeError(f"Unfinished extraction files: {unfinished}")
audit["status"] = "complete"
save_audit()
print(f"Verified {len(audit['archives'])} archives and {sum(len(a['files']) for a in audit['archives'])} extracted files.")
print(f"Audit: {report.relative_to(PROJECT_ROOT)}")
print("Use --output data/raw/moon/lola_product90 for future LOLA downloads.")
print("Diviner and illumination output paths are unchanged. Original ZIPs are preserved.")